- 범주형 데이터 변환
  - lavel Encoding
  - One-hot Encoding
- 수치형 데이터 피처 스케일링
  - StandardScaler : 평균 0으로 표준편차를 1로 맞추는
  - MinMaxScaler : 최소값 최대값을 0과 1로 맞추는

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.DataFrame({
    '상품' : ['사과', '바나나', '사과', '체리', '바나나'],
    '등급' : ['A', 'C', 'B', 'A', 'B'],
    '가격' : [1000, 2000, 1500, 3000, 2500]
})

print(df)
#label Encoding
le = LabelEncoder()
df_le = df.copy()
df_le['상품_encoded'] = le.fit_transform(df['상품'])
print(df_le[['상품','상품_encoded']])

#One-Hot
df_ohe = pd.get_dummies(df, columns=['상품', '등급'], drop_first=False) # drop_first = False -> 다중공선성을 처리하지않겠다.(해결하는 방법중 하나.) -> 원래 그대로 출력
print(df_ohe)                                   #다중공선성 ->회귀분석에서 독립변수들 간에 강한 상관관계가 나타나는 현상
                                                 #True 로바꾸면 다중공선성을 처리함. y값에대해서 영향을 미침  -> 첫번째열을 지웠음.


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num_data = df[['가격']]  #2차원 데이터프레임
#표준화
std_scaler =  StandardScaler()
scaled_std = std_scaler.fit_transform(num_data)
print(scaled_std)
#정규화
min_max_scaler = MinMaxScaler()
scaled_minmax = min_max_scaler.fit_transform(num_data)
print(scaled_minmax)

df_scaled = df[['가격']].copy()
df_scaled['StandardScaler'] = scaled_std  # 표준화 - : 평균이하  0 : 평균 + :평균이상
df_scaled['MinMaxScaler'] = scaled_minmax
print(df_scaled)


수치형과 범주형열이 섞여 있을 때 ColumnTransfomer를 사용하면 한번에 전처리를 완료할 수 있으며, 교차검증(CV)시 데이터 누수를 완벽하게 방지한다.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df[['상품', '가격']]
y = [0, 1, 0 , 1 , 1]
print(df)

num_features = ['가격']
cat_features = ['상품']
#1. 각 컬럼 유형별 전처리 파이프라인 정의(이름, 적용할 전처리 객체, 적용할 칼럼 리스트)
preprocessor = ColumnTransformer(     # ColumnTransformer()는 데이터프레임의 열마다 서로 다른 전처리들을 한번에 적용 할 수 있게 돕는 강력한 도구이다.
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(), cat_features)
    ])
# 2. 전체 머신러닝 파이프라인 생성(전처리 + 모델)
full_pipeline= Pipeline(steps=[
    ('preprocessor', preprocessor),                         #전처리를 실행하고
    ('classifier', DecisionTreeClassifier(random_state=42)) #모델 학습 및 예측을 실행한다.
])
# 3. 학습
full_pipeline.fit(X,y)
print("==========파이프라인 학습완성==========")
print("새로운 데이터 예측 가능 여부:", full_pipeline.predict(X[:2]))
print(f"파이프라인 모델 정확도: {full_pipeline.score(X, y):.2f}")

### 앙상블

- 보팅: 서로 다른 종류의 여러 모델을 모아 투표로 결정
- 배깅: 같은 알고리즘을 여러개 쓰되, 데이터를 조금씩 다르게 쥐여주고 각각 학습을 시킨 뒤 투표하는 방식 (대표적인 예시 : 랜덤 포레스트)
- 부스팅: 틀린 문제에 가중치를 주어 순차적으로 학습하는 방법 (
  (XGBoost, LightGBM)

In [ ]:
from sys import deactivate_stack_trampoline
# 랜덤 포레스트
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

#타이타닉 로드
titanic = sns.load_dataset('titanic')
df = titanic.loc[:, ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch','fare']].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df = pd.get_dummies(df, columns=['sex'], drop_first=True , dtype = int)
X = df.drop(columns = 'survived')
y = df['survived']
# 데이터분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify=y)

# 첫번째 비교군 : 단일 의사결정 나무
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42) #깊이가 깊어지면 과적합 -> 최적의 깊이를 찾아야함.보통 3~5로 설정
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)

# 두번째 비교군 : 랜덤 포레스트
rf_model = RandomForestClassifier(
    max_depth=3,
    random_state=42,
    n_estimators= 100, # 만들 나무의 개수 (기본값 100)
    n_jobs = -1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc =accuracy_score(y_test, rf_pred)

print("============단일 의사 결정 나무 vs 랜덤 포레스트 ===============")
print(f'단일 의사결정나무 정확도:{dt_acc*100:.4f}')
print(f'랜덤 포레스트 정확도:{rf_acc*100:.2f}')

In [ ]:
filture_imp_df = pd.DataFrame({
    'feature':X_train.columns,
    'importance':rf_model.feature_importances_
}).sort_values('importance', ascending=False)
print(filture_imp_df)

## 그래디언트 부스팅(Gradient Boosting)
- 오답노트를 반복하여 실력을 키우는 부스팅
- 배깅과 부스팅의 차이
  - 배깅: 여러 명의 학생이 각자 독립적으로 문제를 풀고 마지막에 다수결로 정답을 매긴다.
  - 부스팅: 1번 학생이 푼 시험지에서 틀린 문제만 골라내어 2번 학생이 그 틀린 문제를 집중해서 다시 푼다. 이과정을 순차적으로 반복하여 전체 실력을 극한으로 끌어올린다.
- XGBoost/LigthGBM
  - XGBoost : GBM의 느린 속도를 개선하고 과적합 방지 규제를 추가하여 정확도와 속도를 모두 잡은 전설적인 알고리즘
  - LightGBM : 마이크로 소프트에서 개발한 모델로 데이터를 줄세워 탐색하는 기존방식 대신 리프중심 나무 분할 방식을 사용하여 대용량 데이터에서도 압도적인 빠른 학습을 자랑함.

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
#타이타닉 로드
titanic = sns.load_dataset('titanic')
#print(titanic)
df = titanic.loc[:, ['survived','pclass','sex','age','sibsp','parch','fare']].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df = pd.get_dummies(df, columns=['sex'], drop_first=True, dtype=int)
X = df.drop(columns='survived')
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# XGBoost 모델 학습 및 평가
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    n_estimators=100,         #의사 결정 나무의 총 개수, 적으면 과소적합, 크면 과적합
    learning_rate=0.1,        #학습률 (0.01~0.1)
    max_depth=3,
    random_state=42,
    eval_metric='logloss')    #로그손실 : 이진 분류 모델의 성능을 평가하는 손실함수
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
print(f"[XGBoost 정확도]: {accuracy_score(y_test, xgb_pred)*100:.2f}")

# LightGBM 모델 학습 및 평가
from lightgbm import LGBMClassifier
lgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    vervose= -1            #텍스트
)
lgbm_model.fit(X_train, y_train)
lgbm_pred = lgbm_model.predict(X_test)
print(f"[LightGBM 정확도]: {accuracy_score(y_test, lgbm_pred)*100:.2f}")

로지스틱 회귀와 KNN분류
- 로지스틱 회귀는 분류모델이다.
- 시그모이드 함수(0과 1사이의 확률값으로 압축)
- 최근접 이웃알고리즘(가까운 이웃 데이터 k개를 찾는다)
- KNN의 경우는 거리를 계산하는 알고리즘으로 반드시 피처 스케일링을 거처야한다.
  - 데이터 구조에 따라서 모델들의 평가가 달라진다.

In [ ]:
#로지스틱 회귀 및 KNN 분류
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
#타이타닉 로드
titanic = sns.load_dataset('titanic')
#print(titanic)
df = titanic.loc[:, ['survived','pclass','sex','age','sibsp','parch','fare']].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df = pd.get_dummies(df, columns=['sex'], drop_first=True, dtype=int)
X = df.drop(columns='survived')
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# 거리 기반/선형 모델을 위한 피처 스케일링을 필수 적용
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 로지스틱 회귀 학습 및 평가
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
print(f"============로지스틱 회귀 정확도============:{accuracy_score(y_test, lr_pred)*100:.2f}%========")
#print(classification_report(y_test,lr_pred))

#KNN 학습 및 평가
knn_model = KNeighborsClassifier(n_neighbors = 5)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)
print(f"======== KNN 정확도: {accuracy_score(y_test, knn_pred)* 100:.2f}%========")

하이퍼파라미터 튜닝 심화
- GridSearchCV의 한계를 극복하는 RandomizedSearchCV

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
#타이타닉 로드
titanic = sns.load_dataset('titanic')
#print(titanic)
df = titanic.loc[:, ['survived','pclass','sex','age','sibsp','parch','fare']].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
df = pd.get_dummies(df, columns=['sex'], drop_first=True, dtype=int)
X = df.drop(columns='survived')
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# 탐색할 파라미터 분포 정의
param_distributions = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'learning_rate' : [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6,0.8,1.0],
    'colsample_bytree':[0.6,0.8,1.0]
    }
xgb_search = XGBClassifier(random_state=42, eval_metric ="logloss")
random_search = RandomizedSearchCV(
    estimator=xgb_search,
    param_distributions =param_distributions,
    n_iter=10,
    scoring='f1',
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs = -1
)

random_search.fit(X_train, y_train)
print(f"최적의 파라미터: {random_search.best_params_}")
print(f"최적 교차 검증 f1-score:{random_search.best_score_:.2f}")